In [ ]:
import importlib
from pathlib import Path
import time


import numpy as np
import matplotlib.pyplot as plt

import nn
import plotting

importlib.reload(nn)
importlib.reload(plotting)

from nn import NeuralNetwork, train_classification,  run_cv_ffnn
from data import load_cv_data_fashion_mnist, one_hot, CLASS_NAMES
from plotting import plot_heatmap
from optimizers import Adam
from tensorflow import keras
from tensorflow.keras.datasets import fashion_mnist

from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)
from metrics import accuracy, accuracy_se


project_root = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
figures_dir = project_root / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
(Xtr, ytr), (Xtest, ytest) = load_cv_data_fashion_mnist()

In [ ]:
# Example code of how to use CV function. Here used for hyperparameter search of learning rate
learning_rates = [1e-2, 1e-3, 1e-4]

for lr in learning_rates:
    loss, best_epoch = run_cv_ffnn(
        Xtr, ytr,
        lr=lr,
        hidden_layers=2,
        nodes=50,
        activations="relu",
        epochs=30
    )

    print(f"LR={lr}: val loss={loss:.4f}, best epoch={best_epoch}")

In [ ]:
learning_rates = [1e-2, 1e-3, 1e-4]
lr_results = {}

for lr in learning_rates:
    loss, best_epoch = run_cv_ffnn(
        Xtr, ytr,
        lr=lr,
        hidden_layers=1,
        nodes=100,
        activations="relu",
        epochs=30,
        batch_size=64
    )
    lr_results[lr] = {"loss": loss, "best_epoch": best_epoch}

best_lr = min(lr_results, key=lambda lr: lr_results[lr]["loss"])

print("lr | mean val loss | best epoch")
for lr, result in lr_results.items():
    print(f"{lr:<8g} | {result['loss']:.4f}         | {result['best_epoch']}")
print(f"Best learning rate: {best_lr}")

In [ ]:
activations = ["sigmoid", "relu", "leaky_relu"]
hidden_layer_options = [1, 2, 3]
node_options = [50, 100, 200]
architecture_results = {}

for activation in activations:
    for hidden_layers in hidden_layer_options:
        for nodes in node_options:
            loss, best_epoch = run_cv_ffnn(
                Xtr, ytr,
                lr=best_lr,
                hidden_layers=hidden_layers,
                nodes=nodes,
                activations=activation,
                epochs=30,
                batch_size=64
            )
            architecture_results[(activation, hidden_layers, nodes)] = {
                "loss": loss,
                "best_epoch": best_epoch
            }

best_act, best_layers, best_nodes = min(
    architecture_results,
    key=lambda key: architecture_results[key]["loss"]
)

print("activation | hidden layers | nodes | mean val loss | best epoch")
for (activation, hidden_layers, nodes), result in architecture_results.items():
    print(
        f"{activation:<11} | {hidden_layers:<13} | {nodes:<5} | "
        f"{result['loss']:.4f}         | {result['best_epoch']}"
    )
print(f"Best architecture: {best_act}, {best_layers} hidden layers, {best_nodes} nodes")

In [ ]:
heatmap_values = np.array([
    [architecture_results[(best_act, hidden_layers, nodes)]["loss"] for nodes in node_options]
    for hidden_layers in hidden_layer_options
])

fig, ax = plt.subplots(figsize=(8, 6))
plot_heatmap(
    heatmap_values,
    x_labels=node_options,
    y_labels=hidden_layer_options,
    x_name="Nodes",
    y_name="Hidden layers",
    title="FFNN: min mean val loss, 5-fold CV, 10,000 tr.im.",
    fmt=".4f",
    ax=ax
)
fig.savefig(
    figures_dir / "FFNN_CV_architecture.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
l2_values = [0.0, 1e-4, 1e-3, 1e-2]
l2_results = {}

for l2 in l2_values:
    loss, best_epoch = run_cv_ffnn(
        Xtr, ytr,
        lr=best_lr,
        hidden_layers=best_layers,
        nodes=best_nodes,
        activations=best_act,
        l2=l2,
        epochs=30,
        batch_size=64
    )
    l2_results[l2] = {"loss": loss, "best_epoch": best_epoch}

best_l2 = min(l2_results, key=lambda l2: l2_results[l2]["loss"])
best_final_epoch = l2_results[best_l2]["best_epoch"]
best_params = {
    "lr": best_lr,
    "activation": best_act,
    "hidden_layers": best_layers,
    "nodes": best_nodes,
    "l2": best_l2,
    "best_final_epoch": best_final_epoch
}

print("lambda | mean val loss | best epoch")
for l2, result in l2_results.items():
    print(f"{l2:<8g} | {result['loss']:.4f}         | {result['best_epoch']}")
print("Best parameters:", best_params)

In [ ]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
mean, std = x_train.mean(), x_train.std()

X_train = ((x_train - mean) / std).astype("float32").reshape(len(x_train), 784)
X_test = ((x_test - mean) / std).astype("float32").reshape(len(x_test), 784)
y_train_oh = one_hot(y_train)

model = NeuralNetwork(
    layer_sizes=[784] + [best_nodes] * best_layers + [10],
    activations=[best_act] * best_layers + ["softmax"],
    cost="cross_entropy",
    l1=0.0,
    l2=best_l2,
    seed=2026
)
optimizer = Adam(learning_rate=best_lr)

In [ ]:
start_time = time.perf_counter()
train_classification(
    model,
    optimizer,
    X_train,
    y_train_oh,
    epochs=best_final_epoch,
    batch_size=64,
    seed=2026
)
training_time = time.perf_counter() - start_time

In [ ]:
probs = model.forward(X_test)
y_pred = probs.argmax(axis=1)

acc = accuracy(y_test, y_pred)
se = accuracy_se(y_test, y_pred)
print(f"Test accuracy: {acc:.4f} ? {se:.4f}")
print(f"Training time: {training_time:.2f} seconds")
parameter_count = sum(W.size + b.size for W, b in model.params)
print(f"Parameter count: {parameter_count}")
print(classification_report(
    y_test,
    y_pred,
    target_names=CLASS_NAMES,
    digits=4
))

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=CLASS_NAMES,
    ax=ax[0],
    xticks_rotation=90,
    cmap="Blues"
)
ax[0].set_title("Confusion matrix")

for i, name in enumerate(CLASS_NAMES):
    RocCurveDisplay.from_predictions(
        (y_test == i).astype(int),
        probs[:, i],
        name=name,
        ax=ax[1]
    )
ax[1].set_title("One-vs-rest ROC curves")
ax[1].set_xlabel("False Positive Rate")
ax[1].set_ylabel("True Positive Rate")
ax[1].legend(fontsize=8)

plt.tight_layout()
fig.savefig(
    figures_dir / "FFNN_CV_final_results.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()